# linkp — `link_arrows` and `link_size`

Arrowheads are drawn at each link's destination when `link_arrows=True` (the default is `False`).

There is **no dedicated arrow-size parameter** — `link_size` controls it, through:

```
arrow_len = max(3.2 * stroke_width, 6.0)   # px; the head is as wide as it is long
```

(`LinkP._ARROW_LEN_FACTOR_` / `LinkP._ARROW_LEN_MIN_` in `linkp.py`.) Loosely follows Jenny et al.
2017 §3.3 ("the default length and width parameters of arrowheads are 1.6 times the flow width"),
enlarged well past the paper's literal factor for on-screen legibility at typical linkp stroke
widths; the floor stands in for their separate rule that thin flows get their arrowheads enlarged
for readability.

The consequence to know: **the floor wins until the stroke is wider than 1.875px** (6.0 / 3.2), so
every link at or below `link_size='small'` draws the same 6px head regardless of the data.

In [ ]:
import pathlib, sys
_root = pathlib.Path('..').resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl
from polars2svg import Polars2SVG
from IPython.display import HTML, display

p2s = Polars2SVG()


def show_row(*items, df=None):
    """Render (label, viz) pairs side-by-side in a single HTML table row."""
    cells = ''.join(
        f'<td style="padding:6px 10px;vertical-align:top;text-align:center">'
        f'<div style="font-size:11px;color:#888;margin-bottom:4px">{label}</div>'
        f'{viz._repr_svg_()}</td>'
        for label, viz in items
    )
    if df is not None:
        display(df)
    display(HTML(f'<table><tr>{cells}</tr></table>'))

---
## The graph

Four nodes on a square with a clockwise cycle `a → b → c → d → a`, plus one diagonal `a → c`.
Each edge is repeated a different number of times so `count` varies — that only matters for
`link_size='vary'` at the end.

In [ ]:
edges = ([('a', 'b')] * 1 + [('b', 'c')] * 3 + [('c', 'd')] * 6 +
         [('d', 'a')] * 10 + [('a', 'c')] * 2)
df    = pl.DataFrame({'fm': [e[0] for e in edges], 'to': [e[1] for e in edges]})
pos   = {'a': [0, 1], 'b': [1, 1], 'c': [1, 0], 'd': [0, 0]}
rels  = [('fm', 'to')]

display(df.group_by(['fm', 'to']).len().sort('fm', 'to'))

wxh = (190, 190)
show_row(
    ('link_arrows=False (default)', p2s.linkp(df, rels, pos, wxh=wxh, draw_node_labels=True)),
    ('link_arrows=True',            p2s.linkp(df, rels, pos, wxh=wxh, draw_node_labels=True,
                                              link_arrows=True)),
)

---
## Named sizes

`link_size` resolves through `{'nil': 0.2, 'small': 1, 'medium': 3, 'large': 5}`.

Note that **`nil` and `small` draw the same size arrow**: their strokes (0.2px and 1px) are both
under the 1.875px threshold, so the 6px floor decides. Only `medium` and `large` are actually
proportional to the stroke.

In [ ]:
kw = dict(pos=pos, wxh=wxh, link_arrows=True, draw_node_labels=True)
show_row(
    ("'nil'    stroke 0.2 → arrow 6.0",  p2s.linkp(df, rels, link_size='nil',    **kw)),
    ("'small'  stroke 1.0 → arrow 6.0",  p2s.linkp(df, rels, link_size='small',  **kw)),
    ("'medium' stroke 3.0 → arrow 9.6",  p2s.linkp(df, rels, link_size='medium', **kw)),
    ("'large'  stroke 5.0 → arrow 16.0", p2s.linkp(df, rels, link_size='large',  **kw)),
)

---
## Numeric sizes

A number is used as the stroke width directly. `0.5` and `1.5` are both below the threshold and
land on the same 6px floor; above it the head grows linearly.

In [ ]:
show_row(
    ('link_size=0.5 → arrow 6.0',  p2s.linkp(df, rels, link_size=0.5, **kw)),
    ('link_size=1.5 → arrow 6.0',  p2s.linkp(df, rels, link_size=1.5, **kw)),
    ('link_size=4   → arrow 12.8', p2s.linkp(df, rels, link_size=4,   **kw)),
    ('link_size=8   → arrow 25.6', p2s.linkp(df, rels, link_size=8,   **kw)),
)

### Where the floor stops mattering

`3.2 * w = 6.0` at `w = 1.875`, so that is exactly where the arrow starts tracking the stroke
(unchanged from before the size bump, since factor and floor were scaled together). Sweeping
across it, the first three arrows are identical and the last two are not:

In [ ]:
show_row(*[(f'link_size={w} → arrow {max(3.2 * w, 6.0):.2f}',
            p2s.linkp(df, rels, link_size=w, **kw))
           for w in (1.0, 1.5, 1.875, 2.5, 3.5)])

---
## `link_size='vary'`

Stroke width is interpolated per link over `link_size_range` (default `(0.25, 4)`) by each link's
`count`, so **arrows vary per link too**. With the default range only the heaviest links clear the
1.875px threshold — the lighter ones sit on the 6px floor, which is close to what the paper's
"enlarge the thin ones" rule intends. Widening the range pushes more links past it.

Counts here: `a→b`=1, `a→c`=2, `b→c`=3, `c→d`=6, `d→a`=10 — so `d→a` is the heaviest link.

In [ ]:
show_row(
    ('vary, range=(0.25, 4) [default]', p2s.linkp(df, rels, link_size='vary', **kw)),
    ('vary, range=(1, 10)',             p2s.linkp(df, rels, link_size='vary',
                                                  link_size_range=(1, 10), **kw)),
    ('vary, range=(0.25, 1.5) all floored',
                                        p2s.linkp(df, rels, link_size='vary',
                                                  link_size_range=(0.25, 1.5), **kw)),
)

### The same numbers, computed

The rule is small enough to just evaluate — this is what the renderer does per link:

In [ ]:
def arrow_len(stroke_w):
    return max(3.2 * stroke_w, 6.0)

_named_ = {'nil': 0.2, 'small': 1.0, 'medium': 3.0, 'large': 5.0}
display(pl.DataFrame({
    'link_size':  list(_named_.keys()) + ['0.5', '1.875', '4', '8'],
    'stroke_w':   list(_named_.values()) + [0.5, 1.875, 4.0, 8.0],
}).with_columns(
    pl.col('stroke_w').map_elements(arrow_len, return_dtype=pl.Float64).alias('arrow_len'),
    (pl.col('stroke_w') * 3.2 < 6.0).alias('floored'),
))

---
## Arrows across the three link shapes

The arrow direction is the link's **arrival tangent**: the baseline for `'line'`, and the curve's
end tangent for `'curve'` and `'flowmap'`. The tip is pulled back by the node radius so it lands on
the node's edge rather than under it. Sizing is identical in all three.

In [ ]:
show_row(*[(f"link_shape={s!r}", p2s.linkp(df, rels, link_shape=s, link_size='medium', **kw))
           for s in ('line', 'curve', 'flowmap')])

`node_size` shifts where the tip lands (it does not change the head's size) — with larger nodes the
arrows stop further out:

In [ ]:
show_row(*[(f'node_size={n!r}',
            p2s.linkp(df, rels, pos=pos, wxh=wxh, link_arrows=True,
                      link_size='medium', node_size=n))
           for n in ('nil', 'small', 'medium', 'large')])